# Exam: Time Series Visualization with Bokeh

This exam tests your ability to visualize time series data using the Bokeh library.
You will be working with the **Daily Minimum Temperatures in Melbourne** dataset.

**Dataset:** `daily-minimum-temperatures-in-melbourne.csv`

The dataset contains daily minimum temperatures (in Celsius) recorded in Melbourne, Australia,
from 1981 to 1990.

In [3]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import (
    ColumnDataSource, HoverTool, DatetimeTickFormatter,
    NumeralTickFormatter, DateRangeSlider, CustomJS, Span
)
from bokeh.layouts import row, column
from bokeh.transform import factor_cmap
from bokeh.palettes import Spectral11, Category20

output_notebook()

# Load the Dataset
df = pd.read_csv('../datasets/daily-minimum-temperatures-in-melbourne.csv')
df.columns = ['Date', 'Temperature']
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')
df = df.dropna()

print(f'Dataset loaded: {df.shape[0]} rows')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Temperature range: {df["Temperature"].min()}C to {df["Temperature"].max()}C')
df.head()

Loading BokehJS ...

Dataset loaded: 3650 rows
Date range: 1981-01-01 to 1990-12-31
Temperature range: 0.0C to 26.3C


,Date,Temperature
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


## Question 1: Basic Time Series Line Plot

Create a basic line plot showing the daily minimum temperature over time.
- Use the `Date` column on the x-axis and the `Temperature` column on the y-axis.
- Set the plot title to **"Daily Minimum Temperatures"**.
- Label the x-axis as `Date` and the y-axis as `Temperature (°C)`.
- Add tooltips to display the date and temperature when hovering over the line.
- Enable pan, wheel zoom, and reset tools.

In [4]:
# Answer 1: Basic Time Series Line Plot

source_q1 = ColumnDataSource(df)

p1 = figure(
    title='Daily Minimum Temperatures',
    x_axis_label='Date',
    y_axis_label='Temperature (C)',
    x_axis_type='datetime',
    width=900, height=400,
    tools='pan,wheel_zoom,reset,save'
)

p1.line(
    x='Date', y='Temperature',
    source=source_q1,
    line_width=1.2,
    line_color='steelblue',
    alpha=0.8,
    legend_label='Daily Min Temperature'
)

# Tooltips
hover_q1 = HoverTool(
    tooltips=[
        ('Date',        '@Date{%F}'),
        ('Temperature', '@Temperature{0.0} C'),
    ],
    formatters={'@Date': 'datetime'},
    mode='vline'
)
p1.add_tools(hover_q1)

# Formatting
p1.xaxis.formatter = DatetimeTickFormatter(months='%b %Y', years='%Y')
p1.legend.location = 'top_left'
p1.title.text_font_size = '14pt'
p1.xaxis.axis_label_text_font_size = '12pt'
p1.yaxis.axis_label_text_font_size = '12pt'

show(p1)

## Question 2: Rolling Average

Calculate the 30-day rolling average and plot it alongside the original temperature data.
- Create a new column `Rolling_Avg` in the DataFrame.
- Plot both `Temperature` and `Rolling_Avg` on the same plot.
- Use different colors and line styles to distinguish the two.
- Add a legend and tooltips displaying date, temperature, and rolling average.

In [5]:
# Answer 2: Rolling Average

df['Rolling_Avg'] = df['Temperature'].rolling(window=30, center=True).mean()

source_q2 = ColumnDataSource(df)

p2 = figure(
    title='Daily Minimum Temperatures with 30-Day Rolling Average',
    x_axis_label='Date',
    y_axis_label='Temperature (C)',
    x_axis_type='datetime',
    width=900, height=400,
    tools='pan,wheel_zoom,reset,save'
)

# Original temperature line
p2.line(
    x='Date', y='Temperature',
    source=source_q2,
    line_width=1, line_color='lightsteelblue',
    alpha=0.5, legend_label='Daily Temperature'
)

# 30-day rolling average
p2.line(
    x='Date', y='Rolling_Avg',
    source=source_q2,
    line_width=2.5, line_color='crimson',
    line_dash='solid',
    alpha=0.9, legend_label='30-Day Rolling Average'
)

# Tooltips
hover_q2 = HoverTool(
    tooltips=[
        ('Date',           '@Date{%F}'),
        ('Temperature',    '@Temperature{0.0} C'),
        ('Rolling Avg',    '@Rolling_Avg{0.00} C'),
    ],
    formatters={'@Date': 'datetime'},
    mode='vline'
)
p2.add_tools(hover_q2)

p2.xaxis.formatter = DatetimeTickFormatter(months='%b %Y', years='%Y')
p2.legend.location = 'top_left'
p2.legend.click_policy = 'hide'
p2.title.text_font_size = '14pt'

show(p2)

## Question 3: Monthly Box Plots

Create box plots to visualize the distribution of temperatures for each month.
- Extract the month from `Date` and create a `Month` column.
- Use Bokeh's box plot elements (vbar, segment, whiskers).
- Label the x-axis with month names and y-axis with `Temperature (°C)`.
- Add tooltips displaying month and statistical values (min, max, median).

In [6]:
# Answer 3: Monthly Box Plots

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month.apply(lambda x: month_names[x-1])

# Compute box statistics per month
grouped_month = df.groupby('Month')['Temperature']
stats_month = pd.DataFrame({
    'q1':     grouped_month.quantile(0.25),
    'q2':     grouped_month.quantile(0.50),
    'q3':     grouped_month.quantile(0.75),
    'upper':  grouped_month.max(),
    'lower':  grouped_month.min(),
    'mean':   grouped_month.mean(),
}).reset_index()
stats_month['iqr']    = stats_month['q3'] - stats_month['q1']
stats_month['whisker_top']    = (stats_month['q3'] + 1.5 * stats_month['iqr']).clip(upper=stats_month['upper'])
stats_month['whisker_bottom'] = (stats_month['q1'] - 1.5 * stats_month['iqr']).clip(lower=stats_month['lower'])
stats_month['Month_Name'] = stats_month['Month'].apply(lambda x: month_names[x-1])

source_month = ColumnDataSource(stats_month)

p3 = figure(
    title='Monthly Temperature Distribution (Box Plots)',
    x_axis_label='Month',
    y_axis_label='Temperature (C)',
    x_range=month_names,
    width=900, height=450,
    tools='pan,wheel_zoom,reset,save'
)

# Box body (IQR)
p3.vbar(
    x='Month_Name', top='q3', bottom='q1',
    source=source_month, width=0.6,
    fill_color='steelblue', fill_alpha=0.7,
    line_color='navy'
)

# Median line
p3.segment(
    x0='Month_Name', x1='Month_Name',
    y0='q2', y1='q2',
    source=source_month,
    line_color='white', line_width=2
)

# Upper whisker
p3.segment(
    x0='Month_Name', x1='Month_Name',
    y0='q3', y1='whisker_top',
    source=source_month,
    line_color='black'
)

# Lower whisker
p3.segment(
    x0='Month_Name', x1='Month_Name',
    y0='q1', y1='whisker_bottom',
    source=source_month,
    line_color='black'
)

# Whisker caps
p3.rect(
    x='Month_Name', y='whisker_top',
    source=source_month,
    width=0.3, height=0.05, line_color='black'
)
p3.rect(
    x='Month_Name', y='whisker_bottom',
    source=source_month,
    width=0.3, height=0.05, line_color='black'
)

# Tooltips
hover_q3 = HoverTool(
    tooltips=[
        ('Month',   '@Month_Name'),
        ('Median',  '@q2{0.0} C'),
        ('Q1',      '@q1{0.0} C'),
        ('Q3',      '@q3{0.0} C'),
        ('Min',     '@lower{0.0} C'),
        ('Max',     '@upper{0.0} C'),
    ]
)
p3.add_tools(hover_q3)
p3.title.text_font_size = '14pt'

show(p3)

## Question 4: Yearly Box Plots with Color Mapping

Create box plots to visualize the temperature distribution for each year.
- Extract the year from `Date` and create a `Year` column.
- Use `factor_cmap` to color boxes based on median temperature per year.
- Add tooltips with year and statistical values.
- Enable pan, wheel zoom, and reset tools.

In [7]:
# Answer 4: Yearly Box Plots with factor_cmap

df['Year'] = df['Date'].dt.year.astype(str)

# Compute statistics per year
grouped_year = df.groupby('Year')['Temperature']
stats_year = pd.DataFrame({
    'q1':    grouped_year.quantile(0.25),
    'q2':    grouped_year.quantile(0.50),
    'q3':    grouped_year.quantile(0.75),
    'upper': grouped_year.max(),
    'lower': grouped_year.min(),
    'mean':  grouped_year.mean(),
}).reset_index()
stats_year['iqr']            = stats_year['q3'] - stats_year['q1']
stats_year['whisker_top']    = (stats_year['q3'] + 1.5 * stats_year['iqr']).clip(upper=stats_year['upper'])
stats_year['whisker_bottom'] = (stats_year['q1'] - 1.5 * stats_year['iqr']).clip(lower=stats_year['lower'])

years = stats_year['Year'].tolist()
palette = Spectral11[:len(years)]

source_year = ColumnDataSource(stats_year)

p4 = figure(
    title='Yearly Temperature Distribution with Color Mapping',
    x_axis_label='Year',
    y_axis_label='Temperature (C)',
    x_range=years,
    width=900, height=450,
    tools='pan,wheel_zoom,reset,save'
)

# Box body with factor_cmap
p4.vbar(
    x='Year', top='q3', bottom='q1',
    source=source_year, width=0.6,
    fill_color=factor_cmap('Year', palette=palette, factors=years),
    fill_alpha=0.8, line_color='black', line_width=1
)

# Median line
p4.segment(
    x0='Year', x1='Year', y0='q2', y1='q2',
    source=source_year, line_color='white', line_width=2
)

# Whiskers
p4.segment(x0='Year', x1='Year', y0='q3', y1='whisker_top',
           source=source_year, line_color='black')
p4.segment(x0='Year', x1='Year', y0='q1', y1='whisker_bottom',
           source=source_year, line_color='black')

# Whisker caps
p4.rect(x='Year', y='whisker_top',    source=source_year, width=0.3, height=0.05)
p4.rect(x='Year', y='whisker_bottom', source=source_year, width=0.3, height=0.05)

# Mean circle marker
p4.circle(x='Year', y='mean', source=source_year,
          size=8, color='black', legend_label='Mean')

# Tooltips
hover_q4 = HoverTool(
    tooltips=[
        ('Year',   '@Year'),
        ('Median', '@q2{0.0} C'),
        ('Mean',   '@mean{0.00} C'),
        ('Q1',     '@q1{0.0} C'),
        ('Q3',     '@q3{0.0} C'),
        ('Min',    '@lower{0.0} C'),
        ('Max',    '@upper{0.0} C'),
    ]
)
p4.add_tools(hover_q4)
p4.legend.location = 'top_right'
p4.title.text_font_size = '14pt'

show(p4)

## Question 5: Interactive Time Range Selection

Create an interactive line plot where the user can select a specific time range
using a date range slider.
- Create a basic line plot of `Temperature` over `Date`.
- Implement a `DateRangeSlider` to filter the displayed date range.
- Update the plot dynamically based on the selected range.
- Add tooltips and enable pan, wheel zoom, reset tools.

In [8]:
# Answer 5: Interactive Date Range Slider

from bokeh.models import DateRangeSlider, CustomJS

# Full source and filtered source
source_full  = ColumnDataSource(df)
source_filt  = ColumnDataSource(df.copy())

p5 = figure(
    title='Interactive Temperature View - Use Slider to Select Range',
    x_axis_label='Date',
    y_axis_label='Temperature (C)',
    x_axis_type='datetime',
    width=900, height=380,
    tools='pan,wheel_zoom,reset,save'
)

p5.line(
    x='Date', y='Temperature',
    source=source_filt,
    line_width=1.5, line_color='steelblue',
    legend_label='Daily Min Temperature'
)

hover_q5 = HoverTool(
    tooltips=[
        ('Date',        '@Date{%F}'),
        ('Temperature', '@Temperature{0.0} C'),
    ],
    formatters={'@Date': 'datetime'},
    mode='vline'
)
p5.add_tools(hover_q5)
p5.xaxis.formatter = DatetimeTickFormatter(months='%b %Y', years='%Y')
p5.title.text_font_size = '14pt'

# Date Range Slider
start_ts = int(df['Date'].min().timestamp() * 1000)
end_ts   = int(df['Date'].max().timestamp() * 1000)

date_slider = DateRangeSlider(
    title='Select Date Range',
    start=start_ts, end=end_ts,
    value=(start_ts, end_ts),
    step=1,
    width=900
)

# JavaScript callback to filter data based on slider value
callback = CustomJS(args=dict(source_full=source_full, source_filt=source_filt, slider=date_slider), code="""
    const [start, end] = slider.value;
    const full_data = source_full.data;
    const dates  = full_data['Date'];
    const temps  = full_data['Temperature'];

    const new_dates = [];
    const new_temps = [];

    for (let i = 0; i < dates.length; i++) {
        if (dates[i] >= start && dates[i] <= end) {
            new_dates.push(dates[i]);
            new_temps.push(temps[i]);
        }
    }

    source_filt.data = { Date: new_dates, Temperature: new_temps };
    source_filt.change.emit();
""")

date_slider.js_on_change('value', callback)

layout_q5 = column(p5, date_slider)
show(layout_q5)

## Question 6: Time Series Decomposition

Perform a simple time series decomposition to visualize trend and seasonality.
- Resample to monthly frequency and calculate monthly average temperature.
- Use a moving average to estimate the trend component.
- Calculate the seasonal component by subtracting the trend from the monthly data.
- Create three aligned Bokeh plots (original, trend, seasonal) sharing the same x-axis.
- Add tooltips to each plot and enable pan, wheel zoom, reset tools.

In [9]:
# Answer 6: Time Series Decomposition

# Resample to monthly average
df_monthly = df.set_index('Date')['Temperature'].resample('ME').mean().reset_index()
df_monthly.columns = ['Date', 'Temperature']

# Trend: 12-month centered moving average
df_monthly['Trend'] = df_monthly['Temperature'].rolling(window=12, center=True).mean()

# Seasonal: original - trend
df_monthly['Seasonal'] = df_monthly['Temperature'] - df_monthly['Trend']

src_original = ColumnDataSource(df_monthly)
src_trend    = ColumnDataSource(df_monthly.dropna(subset=['Trend']))
src_seasonal = ColumnDataSource(df_monthly.dropna(subset=['Seasonal']))

# Shared x-range for alignment
from bokeh.models import Range1d

# Plot 1: Original monthly data
p6a = figure(
    title='Original Monthly Temperature',
    x_axis_label='Date', y_axis_label='Temperature (C)',
    x_axis_type='datetime', width=900, height=280,
    tools='pan,wheel_zoom,reset'
)
p6a.line(x='Date', y='Temperature', source=src_original,
         line_color='steelblue', line_width=2, legend_label='Monthly Avg')
p6a.circle(x='Date', y='Temperature', source=src_original,
           size=5, fill_color='steelblue', line_color='white')
p6a.add_tools(HoverTool(
    tooltips=[('Date', '@Date{%b %Y}'), ('Avg Temp', '@Temperature{0.00} C')],
    formatters={'@Date': 'datetime'}
))
p6a.xaxis.formatter = DatetimeTickFormatter(months='%b %Y')
p6a.legend.location = 'top_left'

# Plot 2: Trend component
p6b = figure(
    title='Trend Component (12-Month Moving Average)',
    x_axis_label='Date', y_axis_label='Temperature (C)',
    x_axis_type='datetime', x_range=p6a.x_range,
    width=900, height=280,
    tools='pan,wheel_zoom,reset'
)
p6b.line(x='Date', y='Trend', source=src_trend,
         line_color='crimson', line_width=2.5, legend_label='Trend')
p6b.add_tools(HoverTool(
    tooltips=[('Date', '@Date{%b %Y}'), ('Trend', '@Trend{0.00} C')],
    formatters={'@Date': 'datetime'}
))
p6b.xaxis.formatter = DatetimeTickFormatter(months='%b %Y')
p6b.legend.location = 'top_left'

# Plot 3: Seasonal component
p6c = figure(
    title='Seasonal Component (Monthly - Trend)',
    x_axis_label='Date', y_axis_label='Seasonal (C)',
    x_axis_type='datetime', x_range=p6a.x_range,
    width=900, height=280,
    tools='pan,wheel_zoom,reset'
)
p6c.line(x='Date', y='Seasonal', source=src_seasonal,
         line_color='seagreen', line_width=2, legend_label='Seasonal')
# Zero reference line
p6c.segment(
    x0=[df_monthly['Date'].min()], x1=[df_monthly['Date'].max()],
    y0=[0], y1=[0],
    line_color='black', line_dash='dashed', line_width=1
)
p6c.add_tools(HoverTool(
    tooltips=[('Date', '@Date{%b %Y}'), ('Seasonal', '@Seasonal{0.00} C')],
    formatters={'@Date': 'datetime'}
))
p6c.xaxis.formatter = DatetimeTickFormatter(months='%b %Y')
p6c.legend.location = 'top_left'

# Combined layout
layout_q6 = column(p6a, p6b, p6c)
show(layout_q6)